# 8.4 实验二：走廊避障（推理）

> **AirSim 配置**：使用 `settings.json`，如已在运行笔记本3，需重启 AirSim 重置无人机位置。

## 任务目标

让无人机在工业巡检场景中自主向前飞行，利用深度感知避开建筑物和设施。

与悬停任务的区别：悬停是"保持不动"，避障是"边飞边躲"——需要同时处理前进和避障两个目标。

![避障任务场景](figures/avoidance_scene.png)

*图 8-8：AirSim 中的前向飞行场景——无人机需要在建筑物和设施之间自主导航*

![悬停与避障对比](figures/hover_vs_avoidance.png)

*图 8-9：悬停 vs 避障——避障任务需要同时优化前进和安全两个目标*

In [ ]:
import sys
sys.path.append('../external-libraries')

import numpy as np
import matplotlib.pyplot as plt
import airsim
import time
from PIL import Image

client = airsim.MultirotorClient()
client.confirmConnection()
DRONE = "Drone1"
print(f"连接成功！")

## 8.4.1 奖励函数设计

避障任务的奖励函数需要平衡两个目标：

| 条件 | 奖励 | 含义 |
|------|------|------|
| 向前推进一段距离 | +1.0/step | 鼓励前进 |
| 距障碍物过近 | -1.0/step | 惩罚危险接近 |
| 发生碰撞 | -100.0 | 严厉惩罚碰撞 |
| 飞出边界/坠机 | -100.0 | 严厉惩罚失控 |

In [ ]:
START_POS = np.array([0.0, 0.0, -8.0])  # 起始位置，8米高
FORWARD_DIR = np.array([1.0, 0.0, 0.0])  # 前进方向（北）

def get_depth_image(client, drone_id):
    """获取深度图，归一化到 [0,1]。"""
    responses = client.simGetImages([
        airsim.ImageRequest("0", airsim.ImageType.DepthPlanar, True)
    ], vehicle_name=drone_id)
    if responses and responses[0].width > 0:
        depth = airsim.list_to_2d_float_array(
            responses[0].image_data_float, responses[0].width, responses[0].height
        )
        return np.clip(depth / 50.0, 0, 1)  # 50米范围归一化
    return np.zeros((64, 64))

def get_rgb_image(client, drone_id):
    """获取 RGB 图像。"""
    responses = client.simGetImages([
        airsim.ImageRequest("0", airsim.ImageType.Scene, False, False)
    ], vehicle_name=drone_id)
    if responses and responses[0].width > 0:
        img = np.frombuffer(responses[0].image_data_uint8, dtype=np.uint8)
        return img.reshape(responses[0].height, responses[0].width, 3)
    return np.zeros((64, 64, 3), dtype=np.uint8)

print("传感器函数定义完成")

## 8.4.2 基于深度图的避障策略

在没有预训练 DreamerV3 权重的情况下，我们实现一个基于深度图的反应式避障策略，作为 baseline 和演示。

核心思路：将深度图分为左、中、右三个区域，哪边更远就往哪边偏。

![深度图示意](figures/depth_scene.png)

*图 8-10：深度图将"前方物体距离"编码为像素强度——亮色表示远，暗色表示近*

In [ ]:
def depth_based_policy(client, drone_id):
    """基于深度图的反应式避障策略。"""
    depth = get_depth_image(client, drone_id)
    h, w = depth.shape

    # 将深度图分为左、中、右三个区域
    left_dist = depth[:, :w//3].mean()
    center_dist = depth[:, w//3:2*w//3].mean()
    right_dist = depth[:, 2*w//3:].mean()

    min_depth = depth[h//4:3*h//4, w//4:3*w//4].min()  # 中心区域最近距离

    # 基本前进
    pitch = 0.3  # 向前倾斜
    roll = 0.0
    yaw_rate = 0.0
    throttle = 0.1

    # 如果前方太近，减速并转向
    if min_depth < 0.15:  # 约7.5米内有障碍
        pitch = 0.0  # 停止前进
        if left_dist > right_dist:
            yaw_rate = -0.5  # 左转
        else:
            yaw_rate = 0.5   # 右转
    elif min_depth < 0.3:  # 约15米内有障碍
        pitch = 0.15  # 减速
        if left_dist > right_dist:
            yaw_rate = -0.2
        else:
            yaw_rate = 0.2

    return np.array([pitch, roll, yaw_rate, throttle]), {
        'left': left_dist, 'center': center_dist, 'right': right_dist, 'min': min_depth
    }

print("深度避障策略定义完成")

In [ ]:
# 运行避障实验
client.reset()
client.enableApiControl(True, vehicle_name=DRONE)
client.armDisarm(True, vehicle_name=DRONE)
client.takeoffAsync(vehicle_name=DRONE).join()
client.moveToPositionAsync(START_POS[0], START_POS[1], START_POS[2], 3, vehicle_name=DRONE).join()
time.sleep(1)

positions = []
rewards = []
depth_snapshots = []
rgb_snapshots = []
prev_pos = START_POS.copy()

N_STEPS = 300
print(f"开始避障飞行，共 {N_STEPS} 步...")

for step in range(N_STEPS):
    action, depth_info = depth_based_policy(client, DRONE)

    client.moveByRollPitchYawrateThrottleAsync(
        float(action[1]) * 0.3,
        float(action[0]) * 0.3,
        float(action[2]) * 0.5,
        float(action[3]) * 0.5 + 0.5,
        duration=0.1, vehicle_name=DRONE
    ).join()

    state = client.getMultirotorState(vehicle_name=DRONE)
    pos = np.array([state.kinematics_estimated.position.x_val,
                    state.kinematics_estimated.position.y_val,
                    state.kinematics_estimated.position.z_val])

    # 计算奖励
    collision = client.simGetCollisionInfo(vehicle_name=DRONE)
    if collision.has_collided:
        print(f"  碰撞！step={step}")
        rewards.append(-100)
        positions.append(pos)
        break

    forward_progress = np.dot(pos - prev_pos, FORWARD_DIR)
    reward = forward_progress * 2.0
    if depth_info['min'] < 0.1:
        reward -= 1.0
    rewards.append(reward)
    positions.append(pos)
    prev_pos = pos.copy()

    # 每50步保存一次快照
    if step % 50 == 0:
        depth_snapshots.append(get_depth_image(client, DRONE))
        rgb_snapshots.append(get_rgb_image(client, DRONE))
        print(f"  step {step}: pos=({pos[0]:.1f},{pos[1]:.1f},{pos[2]:.1f}), min_depth={depth_info['min']:.2f}")

positions = np.array(positions)
rewards = np.array(rewards)
total_forward = np.dot(positions[-1] - START_POS, FORWARD_DIR)
print(f"\n飞行完成！前进距离: {total_forward:.1f}m, 碰撞: {'是' if collision.has_collided else '否'}")

In [ ]:
# 可视化飞行轨迹和深度图
fig = plt.figure(figsize=(16, 8))

# 俯视图轨迹
ax1 = fig.add_subplot(2, 2, 1)
ax1.plot(positions[:, 0], positions[:, 1], 'b-', linewidth=1.5)
ax1.plot(positions[0, 0], positions[0, 1], 'go', markersize=10, label='起点')
ax1.plot(positions[-1, 0], positions[-1, 1], 'r*', markersize=12, label='终点')
ax1.set_xlabel('X (北)'); ax1.set_ylabel('Y (东)')
ax1.set_title('飞行轨迹（俯视图）'); ax1.legend(); ax1.grid(True, alpha=0.3)

# 高度变化
ax2 = fig.add_subplot(2, 2, 2)
ax2.plot(-positions[:, 2], 'b-')  # NED转正常高度
ax2.set_xlabel('时间步'); ax2.set_ylabel('高度 (m)')
ax2.set_title('飞行高度'); ax2.grid(True, alpha=0.3)

# 深度图快照
if depth_snapshots:
    ax3 = fig.add_subplot(2, 2, 3)
    ax3.imshow(depth_snapshots[0], cmap='viridis')
    ax3.set_title('深度图（起始）'); ax3.axis('off')

    ax4 = fig.add_subplot(2, 2, 4)
    ax4.imshow(depth_snapshots[-1], cmap='viridis')
    ax4.set_title(f'深度图（step {(len(depth_snapshots)-1)*50}）'); ax4.axis('off')

plt.tight_layout()
plt.savefig('avoidance_result.png', dpi=150)
plt.show()

In [ ]:
# 保存 RGB 快照拼图
if rgb_snapshots:
    n = min(len(rgb_snapshots), 4)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1:
        axes = [axes]
    for i in range(n):
        axes[i].imshow(rgb_snapshots[i][:,:,::-1])  # BGR to RGB
        axes[i].set_title(f'Step {i*50}')
        axes[i].axis('off')
    plt.suptitle('无人机视角快照', fontsize=14)
    plt.tight_layout()
    plt.savefig('avoidance_snapshots.png', dpi=150)
    plt.show()

## 8.4.3 世界模型的"想象"可视化

世界模型最独特的能力是"想象未来"。下面我们用简化世界模型展示这个过程：给定当前状态和一个动作序列，模型预测未来几步的状态变化。

In [ ]:
import torch
from world_model_tools import SimpleWorldModel

# 用简化世界模型演示"想象"过程
wm = SimpleWorldModel(state_dim=6, action_dim=4)

# 当前状态
state = client.getMultirotorState(vehicle_name=DRONE)
pos = state.kinematics_estimated.position
vel = state.kinematics_estimated.linear_velocity
current = torch.tensor([[pos.x_val, pos.y_val, pos.z_val,
                          vel.x_val, vel.y_val, vel.z_val]])

# 想象3种不同的动作序列
scenarios = {
    '直飞': torch.tensor([[0.3, 0.0, 0.0, 0.1]]),
    '左转': torch.tensor([[0.1, 0.0, -0.5, 0.1]]),
    '右转': torch.tensor([[0.1, 0.0, 0.5, 0.1]]),
}

fig, ax = plt.subplots(figsize=(8, 6))
colors = {'直飞': 'blue', '左转': 'green', '右转': 'red'}

with torch.no_grad():
    for name, action in scenarios.items():
        s = current.clone()
        traj = [s.numpy()[0, :3]]
        for _ in range(20):
            s, r = wm(s, action)
            traj.append(s.numpy()[0, :3])
        traj = np.array(traj)
        ax.plot(traj[:, 0], traj[:, 1], f'{colors[name][0]}-o',
                markersize=3, label=f'想象: {name}', alpha=0.7)

ax.plot(current[0, 0].item(), current[0, 1].item(), 'k*', markersize=15, label='当前位置')
ax.set_xlabel('X (北)'); ax.set_ylabel('Y (东)')
ax.set_title('世界模型的"想象"：预测不同动作的未来轨迹')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('imagination_visualization.png', dpi=150)
plt.show()
print("注意：这是未训练的简化模型，轨迹不准确。")
print("训练后的 DreamerV3 能够准确预测真实环境的动态变化。")

## 8.4.4 小结

本节展示了：

1. **深度图**是避障的关键传感器输入——它直接告诉无人机"前方多远有障碍"
2. 基于深度图的**反应式策略**可以实现基本避障，但缺乏前瞻性
3. 世界模型的核心优势是**"想象"**——在脑中模拟不同动作的后果，选择最安全的路径
4. 预训练的 DreamerV3 能够学会更复杂的避障策略（需要 GPU 训练，见后续章节）

下一节（8.5），我们将在 GPU 机器上从零训练悬停任务的 DreamerV3 模型。